In [2]:

from pathlib import Path
import json

import pandas as pd
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions


def find_project_root(start: Path) -> Path:
    """Walk up from the current folder until we find the repo root."""
    for candidate in (start, *start.parents):
        if (candidate / "data" / "input").is_dir():
            return candidate
    return start


# 1. Paths
PROJECT_ROOT = find_project_root(Path.cwd())
PDF_PATH = PROJECT_ROOT / "data" / "input" / "sample_input_1.pdf"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "structured"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH}\n"
        "Update PDF_PATH to the actual location of sample_input_1.pdf."
    )


# 2. Configure document parsing
# OCR is disabled for this first test because native text was found
# on every page in the previous inspection.
# Table structure detection remains enabled.
pipeline_options = PdfPipelineOptions(
    artifacts_path=Path(r"D:\DoclingModels"),
    do_ocr=False,
    do_table_structure=True,
    generate_picture_images=True,
    generate_page_images=False,
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)


# 3. Parse the PDF
print(f"Parsing: {PDF_PATH.name}")
print("The first run may download models and take some time.")

result = converter.convert(str(PDF_PATH))
document = result.document

print(f"Conversion status: {result.status}")


# 4. Save structured JSON
json_path = OUTPUT_DIR / "document_structure.json"

with json_path.open("w", encoding="utf-8") as file:
    json.dump(
        document.export_to_dict(),
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Saved structured JSON: {json_path}")


# 5. Save Markdown
markdown_path = OUTPUT_DIR / "document.md"

markdown_path.write_text(
    document.export_to_markdown(),
    encoding="utf-8",
)

print(f"Saved Markdown: {markdown_path}")


# 6. Extract detected tables to CSV
print(f"\nDetected tables: {len(document.tables)}")

table_summary = []

for index, table in enumerate(document.tables, start=1):
    dataframe = table.export_to_dataframe(doc=document)

    csv_path = TABLES_DIR / f"table_{index:03}.csv"
    dataframe.to_csv(csv_path, index=False)

    table_summary.append({
        "table_number": index,
        "rows": int(dataframe.shape[0]),
        "columns": int(dataframe.shape[1]),
        "csv_path": str(csv_path.relative_to(PROJECT_ROOT)),
    })

    print(
        f"Table {index}: {dataframe.shape[0]} rows, "
        f"{dataframe.shape[1]} columns -> {csv_path.name}"
    )

    print(dataframe.head(5).to_string(index=False))
    print("-" * 60)


# 7. Extract detected figure images
print(f"\nDetected figures: {len(document.pictures)}")

figure_summary = []

for index, picture in enumerate(document.pictures, start=1):
    image = picture.get_image(document)

    if image is None:
        print(f"Figure {index}: no image crop available")
        continue

    image_path = FIGURES_DIR / f"figure_{index:03}.png"
    image.save(image_path)

    caption = picture.caption_text(document)

    figure_summary.append({
        "figure_number": index,
        "caption": caption,
        "image_path": str(image_path.relative_to(PROJECT_ROOT)),
    })

    print(f"Figure {index}: saved {image_path.name}")
    if caption:
        print(f"Caption: {caption}")


# 8. Save a compact extraction summary
summary = {
    "source_pdf": PDF_PATH.name,
    "conversion_status": str(result.status),
    "table_count": len(document.tables),
    "tables": table_summary,
    "detected_figure_count": len(document.pictures),
    "saved_figures": figure_summary,
}

summary_path = OUTPUT_DIR / "extraction_summary.json"

summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nStructured extraction finished.")
print(f"Summary: {summary_path}")


Parsing: sample_input_1.pdf
The first run may download models and take some time.


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 2841.26it/s]


Conversion status: ConversionStatus.SUCCESS
Saved structured JSON: d:\Acalvio\DeCraft\outputs\structured\document_structure.json
Saved Markdown: d:\Acalvio\DeCraft\outputs\structured\document.md

Detected tables: 12
Table 1: 2 rows, 2 columns -> table_001.csv
                      0                   1
       BSE CODE: 500400 NSE CODE: TATAPOWER
Bloomberg CODE: TPWR:IN      SENSEX: 67,127
------------------------------------------------------------
Table 2: 23 rows, 4 columns -> table_002.csv
                      0                       1                       2            3
           Company Data            Company Data            Company Data Company Data
        Market Cap (cr)         Market Cap (cr)         Market Cap (cr)   Rs. 86,856
  Enterprise Value (cr)   Enterprise Value (cr)   Enterprise Value (cr)  Rs. 131,339
Outstanding Shares (cr) Outstanding Shares (cr) Outstanding Shares (cr)        319.6
             Free Float              Free Float              Free Float      